[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Pagination


## What you will be able to do

Fetch every item of a list that an API sends in pages, by page number, by following the `Link`
header and by cursor, stop at the last page without fetching pages you do not need, and say why, in
a list that changes, a cursor keeps its place and a page number does not.


## The idea

### The problem

Every list so far fit in one response: `/stations` sends four stations. A list can be far longer,
such as every commit in a repository, every order a shop has taken, or every reading a weather
network has made. Sent whole, it would cost the server time and memory to build, take a long time
to download, and could run past a client's `timeout`. So an API sends a long list a part at a time,
and the client asks for the parts in turn.

That leaves the client two decisions: how to ask for the next part, and when to stop. A loop that
stops too soon loses items without an error. A loop that never stops asks for the same pages again
and again, until someone notices the requests or the bill. The practice API's hourly readings are a
list like that, 216 readings sent 30 to a response.

### What pagination is

> **Pagination** is how an API splits a long list into **pages**, which a client asks for one at a
> time. With **page numbers**, the client names a page by its position, as `page` and `per_page`
> do. With a **cursor**, every response carries a bookmark, which the client sends back to get what
> comes after it. Either way, a response says whether more remains: with a total, with the address
> of the next page in a `Link` header, or with a cursor that is `null` on the last page.

### Why it works that way

- **A server limits every response.** It caps how many items a page may hold, and a client that
  asks for more gets the most the server allows, or an error.
- **A page number counts from the top.** At 30 a page, page 3 is whatever sits in places 61 to 90
  when the request arrives. A client can jump to any page, but an item added or removed while it
  pages moves every item after it, so one item is seen twice, or never.
- **A cursor names a place, not a position.** It marks the last item sent, so a change before that
  item does not move what comes after it. The price is that a client cannot jump ahead: it reads the
  pages in order.
- **The server says when the list ends.** A missing `next` link, a `null` cursor or
  `has_more: false` marks the last page. A page with fewer items than asked for need not be the last.
- **A link beats an address you build.** The `next` address in a `Link` header carries the whole
  query, so a client that follows it keeps working if the API changes how it names its pages.

### Where you will meet this

GitHub's API numbers pages with `page` and `per_page`, at most 100 to a page on most endpoints, and
puts the previous, next, first and last pages in a `Link` header, whose addresses its documentation
says to use. Stripe's API pages with a cursor: `starting_after` takes the id of the last object
received, and `has_more` says whether more remain. Slack's API sends `next_cursor`, and warns that a
page can hold fewer results than the limit while more remain. Anthropic's API sends a `next_page`
cursor that is `null` on the last page. Stripe's and Anthropic's client libraries page through lists
for you, which is easier to trust once you have written the loop yourself.

### What this notebook covers

- One page of a list: its readings, and `page`, `per_page` and `total`
- A page asked for by number, and a page past the end
- A loop over page numbers that knows where it ends before it starts
- The `Link` header, and a loop that follows `next` until there is none
- A class whose `__iter__` fetches a page only when a loop asks for more
- Cursors, and a loop that stops when `next_cursor` is `null`
- A list that changes while a client pages through it, by number and by cursor
- A pager for any list, which follows links or cursors and gives up after a set number of pages
- Five errors, from a query sent twice to a cursor of `None`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

response = requests.get("http://127.0.0.1:8765/network/readings", timeout=10)
body = response.json()

print(len(body["readings"]), "of", body["total"], "readings, on page", body["page"])
print("next:", response.links["next"]["url"])
```

```
30 of 216 readings, on page 1
next: http://127.0.0.1:8765/network/readings?page=2
```

One request, one page of a longer list, and the address of the next page, which the server sent in
the response's headers.


## Setup

Eight imports, the last of them the practice API.

- `requests` sends every request, and a response's `links` holds its `Link` header
- `math` turns a total into a number of pages, with `ceil`
- `itertools` takes the first few items of a long list, with `islice`
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`


In [1]:
import importlib
import itertools
import math
import sys
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### One page of a list

`/network/readings` holds the network's hourly readings for the three days up to the practice API's
clock. A request with no parameters gets the first page:


In [2]:
response = requests.get(f"{BASE}/network/readings", timeout=10)
body = response.json()

print({name: value for name, value in body.items() if name != "readings"})
print("first:", body["readings"][0])
print("last: ", body["readings"][-1])


{'page': 1, 'per_page': 30, 'total': 216}
first: {'station': 'bergen', 'time': '2026-02-26T10:00Z', 'temperature_c': 3.8}
last:  {'station': 'tromso', 'time': '2026-02-26T19:00Z', 'temperature_c': -4.9}


The readings are made up. They run in time order, and within an hour in the order of the stations'
ids, so page 1's thirty readings are the first ten hours at three stations. `Z` at the end of a time
means UTC. `total` counts the whole list: 72 hours at Bergen, Oslo and Tromso. Svalbard has no
readings, because it has been inactive since January 12, as `/network` records. `page` and
`per_page` say which part of the list this is.

### A page asked for by number

`page` asks for a page, counting from 1, and `per_page` sets how many readings a page holds.
`math.ceil` rounds up, which turns a total into a number of pages:


In [3]:
for params in [{"page": 2}, {"page": 8}, {"page": 9}, {"per_page": 100, "page": 3}]:
    body = requests.get(f"{BASE}/network/readings", params=params, timeout=10).json()
    pages = math.ceil(body["total"] / body["per_page"])
    print(f"{str(params):<29} {len(body['readings']):>3} readings, page {body['page']} of {pages}")


{'page': 2}                    30 readings, page 2 of 8
{'page': 8}                     6 readings, page 8 of 8
{'page': 9}                     0 readings, page 9 of 8
{'per_page': 100, 'page': 3}   16 readings, page 3 of 3


Thirty readings a page make eight pages, the last holding the six left over. A page past the end is
not an error: it comes back empty, with a `200`. A hundred a page make three pages, so a client that
needs the whole list asks for the most a page may hold, and sends fewer requests.

### A loop over page numbers

The first page's `total` tells a loop how many pages to ask for before it starts:


In [4]:
first = requests.get(f"{BASE}/network/readings", params={"per_page": 100}, timeout=10).json()
pages = math.ceil(first["total"] / first["per_page"])
every_reading = first["readings"]

for page in range(2, pages + 1):
    response = requests.get(f"{BASE}/network/readings", params={"per_page": 100, "page": page}, timeout=10)
    response.raise_for_status()
    every_reading += response.json()["readings"]

print(pages, "pages,", len(every_reading), "readings, of", first["total"])


3 pages, 216 readings, of 216


`range` ends the loop, so it cannot run on. Not every API sends a total, because counting a long
list costs the server work on every request. Without one, a loop asks for pages until the server
stops offering a next page, or until a page comes back empty, which costs one request for the empty
page.

### Following the Link header

Every page of readings also names its neighbors in a `Link` header, as GitHub's API does. requests
reads that header into `links`, a dictionary keyed by each address's `rel`:


In [5]:
response = requests.get(f"{BASE}/network/readings", params={"page": 4}, timeout=10)

print(response.headers["Link"])
for rel, link in response.links.items():
    print(f"  {rel:<5} {link['url']}")


<http://127.0.0.1:8765/network/readings?page=3>; rel="prev", <http://127.0.0.1:8765/network/readings?page=5>; rel="next", <http://127.0.0.1:8765/network/readings?page=8>; rel="last", <http://127.0.0.1:8765/network/readings?page=1>; rel="first"
  prev  http://127.0.0.1:8765/network/readings?page=3
  next  http://127.0.0.1:8765/network/readings?page=5
  last  http://127.0.0.1:8765/network/readings?page=8
  first http://127.0.0.1:8765/network/readings?page=1


Each address comes with its query already built, so a loop can follow `next` until a page has none.
The first request sends a query, and every request after it sends only the address it was given:


In [6]:
response = requests.get(f"{BASE}/network/readings", params={"station": "tromso", "per_page": 50}, timeout=10)
response.raise_for_status()
tromso = response.json()["readings"]
print(response.url)

while "next" in response.links:
    response = requests.get(response.links["next"]["url"], timeout=10)
    response.raise_for_status()
    tromso += response.json()["readings"]
    print(response.url)

print(len(tromso), "readings at Tromso")


http://127.0.0.1:8765/network/readings?station=tromso&per_page=50
http://127.0.0.1:8765/network/readings?station=tromso&per_page=50&page=2
72 readings at Tromso


The second address kept `station` and `per_page` and added `page`, and the second page had no
`next`, so the loop ended. A client that follows links never builds a page's address itself, as
GitHub's documentation advises: if an API changes how it names its pages, the links change with it,
and the loop does not have to.

### A class that fetches pages only when a loop asks

A loop that collects every page into a list fetches every page, even when the code needs only the
first few items. A class whose `__iter__` holds a `yield`, as in the **Context Managers and
Iterators** notebook, fetches as it goes: the method pauses at every `yield`, and sends the request for the next page only when a
loop has taken the last reading of the page before:


In [7]:
class Readings:
    """The readings for a query, fetched a page at a time, and only when a loop asks for more."""

    def __init__(self, **params):
        self.params = params
        self.pages_fetched = 0

    def __iter__(self):
        response = requests.get(f"{BASE}/network/readings", params=self.params, timeout=10)
        while True:
            response.raise_for_status()
            self.pages_fetched += 1
            for reading in response.json()["readings"]:
                yield reading
            if "next" not in response.links:
                return
            response = requests.get(response.links["next"]["url"], timeout=10)


oslo = Readings(station="oslo")
print(len(list(oslo)), "readings from", oslo.pages_fetched, "pages")

oslo = Readings(station="oslo")
print([reading["time"] for reading in itertools.islice(oslo, 3)], "from", oslo.pages_fetched, "page")


72 readings from 3 pages
['2026-02-26T10:00Z', '2026-02-26T11:00Z', '2026-02-26T12:00Z'] from 1 page


`list` asked for every reading, so all three pages were fetched. `itertools.islice` takes the first
few items of anything a `for` loop can go through and stops asking after them, so the method paused
after the third reading and never sent the request for page 2. A `break` in a `for` loop over
`Readings` saves requests the same way.

### Cursors: a bookmark from the server

`/network/events` is the network's event log, newest first, and it pages with a cursor instead of a
number:


In [8]:
body = requests.get(f"{BASE}/network/events", timeout=10).json()

print(len(body["events"]), "events, the newest:", body["events"][0])
print("next_cursor:", body["next_cursor"])


10 events, the newest: {'id': 54, 'time': '2026-03-01T06:10Z', 'station': 'tromso', 'summary': 'readings uploaded'}
next_cursor: YmVmb3JlOjQ1


The response has no page number and no total, only `next_cursor`, a bookmark the server wrote for
the place in the log after the last event it sent. A client sends it back as `cursor`, exactly as it
arrived, and stops when `next_cursor` is `null`, which `response.json()` gives as `None`:


In [9]:
events, cursor, pages = [], None, 0
while True:
    response = requests.get(f"{BASE}/network/events", params={"cursor": cursor}, timeout=10)
    response.raise_for_status()
    body = response.json()
    events += body["events"]
    pages += 1
    cursor = body["next_cursor"]
    if cursor is None:
        break

print(pages, "pages,", len(events), "events, the oldest:", events[-1])


6 pages, 54 events, the oldest: {'id': 1, 'time': '2025-10-14T09:00Z', 'station': 'bergen', 'summary': 'thermometer and rain gauge calibrated'}


On the first pass `cursor` is `None`, and requests leaves a parameter whose value is `None` out of
the query, so the first request asked for the start of the log. A cursor cannot jump to page 4,
because every cursor comes from the page before it. `limit` sets how many events a page holds, 10
unless asked, and the practice API refuses a limit over 50 with a `400`, where `/network/readings`
quietly sends its most instead. APIs differ on that, which is one more reason to read what comes
back. Stripe's `starting_after` works the same way, with the id of the last object received as the
bookmark.

### A list that changes while a client pages through it

The event log is busy, and new events arrive at the top while a client pages through it.
`/network/events` also answers `page` and `per_page`, and in that mode it adds one new event before
every page after the first, so that the same thing happens on every run. Here are two pages fetched
by number, and two fetched by cursor:


In [10]:
url = f"{BASE}/network/events"
page_1 = requests.get(url, params={"page": 1}, timeout=10).json()["events"]
page_2 = requests.get(url, params={"page": 2}, timeout=10).json()["events"]
first = requests.get(url, timeout=10).json()
second = requests.get(url, params={"cursor": first["next_cursor"]}, timeout=10).json()

by_number = [event["id"] for event in page_1 + page_2]
by_cursor = [event["id"] for event in first["events"] + second["events"]]
print("by number:", len(by_number), "events,", len(set(by_number)), "different, page 2 starts with", by_number[10])
print("by cursor:", len(by_cursor), "events,", len(set(by_cursor)), "different, page 2 starts with", by_cursor[10])


by number: 20 events, 19 different, page 2 starts with 45
by cursor: 20 events, 20 different, page 2 starts with 44


Page 1 ended with event 45. Before page 2 was served, a new event arrived at the top of the log and
every event moved down a place, so the events in places 11 to 20 began with event 45 again. Had an
event been deleted instead, everything would have moved up, and the pages would have skipped an
event. Neither raised an error. The cursor marked the place after event 45, which stayed where it was
however many events arrived above it. That is why lists whose newest items come first, such as logs
and feeds, so often page with cursors. Page numbers suit a list that holds still while it is read,
such as three days of readings already taken.

### A pager for any list, which stops

The pieces of this notebook, in one class. `Paged` takes a list's address, the name of the member
that holds its items, and the query for the first page. It follows a `Link` header when a response
has one and a cursor when a body has one, fetches a page only when a loop asks for more, and raises
an error instead of asking for more than `max_pages` pages:


In [11]:
class Paged:
    """Every item of a paginated list, fetched a page at a time as a loop asks, and never without end."""

    def __init__(self, url, key, params=None, max_pages=20):
        self.url = url
        self.key = key
        self.params = params or {}
        self.max_pages = max_pages
        self.pages_fetched = 0

    def __iter__(self):
        url, params = self.url, self.params
        for _ in range(self.max_pages):
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            self.pages_fetched += 1
            body = response.json()
            for item in body[self.key]:
                yield item
            if "next" in response.links:
                url, params = response.links["next"]["url"], None       # the address carries the query
            elif body.get("next_cursor"):
                params = {**self.params, "cursor": body["next_cursor"]}
            else:
                return
        raise RuntimeError(f"{self.url} had more than {self.max_pages} pages")


readings = Paged(f"{BASE}/network/readings", "readings", {"station": "tromso"})
temperatures = [reading["temperature_c"] for reading in readings]
print(f"Tromso: {len(temperatures)} readings from {readings.pages_fetched} pages, {min(temperatures)} to {max(temperatures)} °C")

events = Paged(f"{BASE}/network/events", "events", {"station": "tromso"})
for event in itertools.islice(events, 3):
    print(f"  {event['time']}  {event['summary']}")
print("the newest three events, from", events.pages_fetched, "page")

try:
    list(Paged(f"{BASE}/network/readings", "readings", max_pages=5))
except RuntimeError as error:
    print("stopped:", error)


Tromso: 72 readings from 3 pages, -8.3 to -3.7 °C
  2026-03-01T06:10Z  readings uploaded
  2026-02-28T06:10Z  readings uploaded
  2026-02-27T10:10Z  maintenance visit booked for 2026-03-23
the newest three events, from 1 page
stopped: http://127.0.0.1:8765/network/readings had more than 5 pages


### Where each part came from

| In the pager | What it relies on | The section that showed it |
|---|---|---|
| `response.links["next"]["url"]`, with `params` set to `None` | the next page's address, with its query already built | Following the Link header |
| `for item in body[self.key]: yield item` | a page fetched only when a loop asks for more | A class that fetches pages only when a loop asks |
| `body.get("next_cursor")`, sent back as `cursor` | a bookmark the server wrote, which is `null` on the last page | Cursors: a bookmark from the server |
| `for _ in range(self.max_pages)` | a loop with a most, which cannot run forever | A loop over page numbers |
| no test of a page's size | the server, not the size of a page, says when a list ends | Cursors: a bookmark from the server |

Tromso's readings took three pages, and its three newest events took one, because `islice` stopped
asking. The last pager gave up after five of the eight pages of readings, which is what a most is
for: a list that keeps offering a next page, through a bug in the server or in the client, costs
five requests and an error, not an endless loop.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/10-pagination-solutions.ipynb).

**1.** Ask `/network/readings` for page 3 with `per_page` set to 50, and print the number of readings
on it and the times of the first and the last.


In [12]:
# your code here


**2.** Work out from `total` how many pages `/network/readings` has for `station` set to `oslo` and
`per_page` set to 25. Check the answer by printing how many readings the last page and the page after
it hold.


In [13]:
# your code here


**3.** Print the `rel` names in the `Link` header of pages 1, 4 and 8 of `/network/readings`.


In [14]:
# your code here


**4.** Follow the `next` links through `/network/readings` for `station` set to `bergen` and
`per_page` set to 40, and print how many requests it took and Bergen's warmest reading.


In [15]:
# your code here


**5.** Page through `/network/events` by cursor with `limit` set to 25, and print the number of pages,
the number of events, and the summary of the oldest event.


In [16]:
# your code here


**6.** Use the `Readings` class, with `per_page` set to 10, and `next` to find the first reading at
Oslo colder than -7 °C. Print it, and how many pages were fetched to find it.


In [17]:
# your code here


## Common errors

### HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:8765/network/readings?station=tromso&per_page=50&page=2&station=tromso&per_page=50


In [18]:
params = {"station": "tromso", "per_page": 50}
first = requests.get(f"{BASE}/network/readings", params=params, timeout=10)
response = requests.get(first.links["next"]["url"], params=params, timeout=10)
response.raise_for_status()


HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:8765/network/readings?station=tromso&per_page=50&page=2&station=tromso&per_page=50

The next page's address already held the query, and `params=` appended the same parameters to it, so
`station` and `per_page` arrived twice. The **Query Parameters** notebook showed that APIs disagree
about a parameter given twice. The practice API refuses one, and says which. Follow a link with no
`params=`:


In [19]:
print(response.json()["error"])

response = requests.get(first.links["next"]["url"], timeout=10)
print(response.status_code, "page", response.json()["page"], "with", len(response.json()["readings"]), "readings")


station was given more than once
200 page 2 with 22 readings


### No error, and 100 readings, not 216: a per_page above the most a page holds


In [20]:
response = requests.get(f"{BASE}/network/readings", params={"per_page": 500}, timeout=10)
body = response.json()

print(len(body["readings"]), "readings, of", body["total"])


100 readings, of 216


Asking for 500 a page was meant to fetch the whole list at once. The practice API sends at most 100
a page, as GitHub's API does on most endpoints, and says so in `per_page` instead of with an error, so
a program that checks neither `per_page` nor the next page loses 116 readings without a sign. Keep asking while
there is a next page:


In [21]:
print("per_page:", body["per_page"], "| a next page:", "next" in response.links)

readings = body["readings"]
while "next" in response.links:
    response = requests.get(response.links["next"]["url"], timeout=10)
    readings += response.json()["readings"]
print(len(readings), "readings")


per_page: 100 | a next page: True
216 readings


### KeyError: 'next'


In [22]:
response = requests.get(f"{BASE}/network/readings", params={"page": 8}, timeout=10)
next_url = response.links["next"]["url"]


KeyError: 'next'

Page 8 is the last, so its `Link` header names no next page, and `links` has no `next` key. Test for
the key with `in`, as the loops above do, or look it up with `get`, which returns `None`:


In [23]:
print(list(response.links), response.links.get("next"))


['prev', 'first'] None


### No error, and events missing: a loop that stopped at a short page


In [24]:
svalbard, cursor = [], None
while True:
    body = requests.get(f"{BASE}/network/events", params={"station": "svalbard", "cursor": cursor}, timeout=10).json()
    svalbard += body["events"]
    cursor = body["next_cursor"]
    if len(body["events"]) < 10:          # fewer than the limit, so it must be the last page
        break

print("events at Svalbard:", len(svalbard))


events at Svalbard: 1


Svalbard has three events in the log, and the loop found one. With `station`, a page looks through
its 10 events and sends the ones at that station, so a page can be short, or even empty, with more to
come, and the first page held one event. Slack's documentation warns of the same thing: a page can
hold fewer results than the limit while more remain. Only `next_cursor` says whether the list has
ended:


In [25]:
svalbard, cursor, sizes = [], None, []
while True:
    body = requests.get(f"{BASE}/network/events", params={"station": "svalbard", "cursor": cursor}, timeout=10).json()
    svalbard += body["events"]
    sizes.append(len(body["events"]))
    cursor = body["next_cursor"]
    if cursor is None:
        break

print("events at Svalbard:", len(svalbard), "| page sizes:", sizes)


events at Svalbard: 3 | page sizes: [1, 0, 0, 0, 0, 2]


### No error, and the first page again: a cursor of None left out of the query


In [26]:
ids, cursor = [], None
for request in range(8):                  # at most 8 requests, so that this cell ends
    body = requests.get(f"{BASE}/network/events", params={"cursor": cursor, "limit": 25}, timeout=10).json()
    ids += [event["id"] for event in body["events"]]
    cursor = body["next_cursor"]

print(len(ids), "events,", len(set(ids)), "different")


158 events, 54 different


The loop never checked for the end. After the third page `cursor` was `None`, requests left it out of
the query, and the practice API answered with the first page, so the loop went round the log again,
and without `range(8)` it would never have stopped. Break when `next_cursor` is `None`, and give any
loop over pages a most, as `Paged` does with `max_pages`:


In [27]:
ids, cursor = [], None
for request in range(8):
    body = requests.get(f"{BASE}/network/events", params={"cursor": cursor, "limit": 25}, timeout=10).json()
    ids += [event["id"] for event in body["events"]]
    cursor = body["next_cursor"]
    if cursor is None:
        break

print(len(ids), "events,", len(set(ids)), "different, from", request + 1, "requests")


54 events, 54 different, from 3 requests


## Recap

- An API sends a long list in pages, and every response says how to ask for more, or that there is
  no more.
- `page` and `per_page` name a page by its position. `total` and `per_page` give the number of pages,
  and a page past the end comes back empty.
- A `Link` header's `next` address carries the whole query: follow it with no `params=`, and stop
  when a response has no `next`.
- A cursor is a bookmark to send back as it arrived. Stop when `next_cursor` is `null`, not when a
  page is short.
- Page numbers repeat or skip items when a list changes between requests, and a cursor does not.
- A class whose `__iter__` yields items fetches a page only when a loop asks, so stopping early saves
  requests. Give every loop over pages a most.


## What is next

The **Rate Limits** notebook. Every page here was sent as soon as it was asked for. That notebook
meets a server that answers `429 Too Many Requests` to a client that asks too often, reads the
headers that say how many requests are left, and waits before asking again.


---

&#8592; **Previous:** [Authentication](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/09-authentication.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
